# ChuckleNet — FINAL Colab Pipeline v20 (2026-09-08)
**Labels: VTT caption laughter markers (Tier-1 weak) · Features: WavLM 768 + prosody 23 · Eval: 5-fold GroupKFold (video-level) F1 + IoU-F1@0.2**

## HOW TO RUN
1. **Runtime → Change runtime type → GPU (T4)** — then *Run all*.
2. Data is already on your Drive: `MyDrive/chuckle_net_1000/` (621 m4a + vtt/). Nothing to upload.
3. **FIRST RUN = GATE MODE** (`GATE_N = 20` below): processes only 20 videos (~5 min) to verify the pipeline end-to-end.
4. If gate passes (features extracted, pos_rate sane, IoU cell runs): set `GATE_N = 0` and *Run all* again → full 621-video run (~2-3 hrs on T4). Checkpoints resume automatically.
5. Results land in `MyDrive/chuckle_net_results/`: `utterance_features_v20.npz` (with per-utterance timestamps), `fusion_model_final.pt`, `results_v20.json`.

## HARD RULES BUILT IN
- **pos_rate < 10% → HARD FAIL** (the Sep-5 disaster: 1.6% positive labels produced garbage)
- **Fresh checkpoint namespace** (`*_v20`) — never loads stale old-format checkpoints
- **Video-level splits only** (GroupKFold on video IDs) — utterance leakage is banned
- **Anchor rule:** these are caption-marker numbers. Report them as such; gold claims (Gillick/StandUp4AI) need separate anchor evals.
- **NO DELETES:** checkpoints/results are additive; nothing is overwritten except within `*_v20` namespace.

---
**v20.1 FIX (Sep 8):** guarded ffmpeg (auto-retry + imageio-ffmpeg fallback — was crashing FileNotFoundError), single OUTPUT dir, new DIAGNOSTIC cell that pinpoints missing-file causes (wrong Google account / missing folders) with exact remediation steps.

---
**v20.2 (Sep 8, post-gate-failure):** First gate run correctly hard-failed (0.7% pos). Forensics: full 620-video set has only **1.16%** marker-positive utterances (2816/243,501 — labels VERIFIED correct, VTTs byte-identical to curated local set; the collection is simply marker-poor: only 174/620 videos have any markers). Also: YouTube auto-caption VTTs duplicate every cue (~50% of parsed utterances were doubles — now deduped+OR-merged). Changes: 3-tier pos gate (FAIL<2% broken / WARN 2–10% proceed / PASS≥10%), curated GATE_IDS (marker-rich 20), dual training (FULL-620 + RICH subset), PR-AUC added, checkpoint namespace → `*_v21`, per-file extraction log.

**v20.2c (Sep 8, evening):** gate run died mid-session and lost ~5 videos — the checkpoint only saved every 20 files (= never during a 20-video gate). Now saves every 3 videos in gate mode / every 10 in full mode (`FILES_PER_SAVE`). Re-run resumes from the 1 saved video.

**Sep 8 (v20.2d, full-run config): `GATE_N = 0` pre-set — Run all = full 620-video run (~6-9 h, resumable). Expect WARN at ~1.16% pos: correct, it proceeds.**


In [ ]:
# === SETUP (v20.1 — single OUTPUT, surfaced install errors) ===
from google.colab import drive
drive.mount('/content/drive')
import os
if os.path.isdir('/content/drive/MyDrive'):
    os.chdir('/content/drive/MyDrive')
else:
    raise RuntimeError('Drive mounted but MyDrive missing — did you complete the Google auth popup?')

print('Installing ffmpeg (errors shown, not hidden)...')
r = os.system('apt-get update -qq && apt-get install -y ffmpeg > /tmp/apt.log 2>&1')
if r != 0:
    print(open('/tmp/apt.log').read()[-800:])
!pip install -q soundfile librosa numpy pandas scikit-learn torch transformers tqdm 2>&1 | tail -1

import numpy as np
import glob
from tqdm import tqdm
import torch
import soundfile as sf
import librosa
import re

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

# === v20 CONFIG (single source) ===
GATE_N = 0   # 20 = gate mode (curated marker-rich 20). Set to 0 for FULL 620-video run.
RICH_MIN_PCT = 6.0   # RICH subset: per-video positive%% >= this
RICH_MIN_POS = 10    #   and per-video positives >= this
FILES_PER_SAVE = 3 if GATE_N else 10   # checkpoint every N videos (gate=3: short sessions safe; full=10)
BASE = '/content/drive/MyDrive/chuckle_net_1000'
AUDIO_DIR = f'{BASE}/audio'
VTT_DIR = f'{BASE}/vtt'
OUTPUT = '/content/drive/MyDrive/chuckle_net_results'
os.makedirs(OUTPUT, exist_ok=True)
print(f'GATE_N={GATE_N} | OUTPUT={OUTPUT}')

audio_files = sorted(glob.glob(f'{AUDIO_DIR}/*.m4a'))
print(f'Audio files found: {len(audio_files)}')


In [ ]:
# === DIAGNOSTIC (v20.1) — pinpoints any file-not-found BEFORE it happens ===
import os, glob
print('Mount check :', 'OK' if os.path.isdir('/content/drive/MyDrive') else 'FAILED — redo drive.mount')
print('BASE exists :', os.path.isdir(f'{BASE}'))

if not os.path.isdir(f'{BASE}'):
    print('\n❌ chuckle_net_1000 NOT FOUND in this Drive.')
    print('Top-level folders actually visible in THIS account:')
    for d in sorted(os.listdir('/content/drive/MyDrive'))[:25]:
        print('   -', d)
    print('\n⚠️ LIKELY CAUSE: Colab is authorized with a DIFFERENT Google account.')
    print('   The data (chuckle_net_1000) lives on the meg...das@gmail.com account.')
    print('   FIX: Runtime → Disconnect and delete runtime → run again → in the auth popup')
    print('        choose the account that owns chuck_net_1000.')
    raise RuntimeError('Wrong Google account / data folder missing — see messages above.')

n_audio = len(glob.glob(f'{AUDIO_DIR}/*.m4a'))
n_vtt = len(glob.glob(f'{VTT_DIR}/*.vtt'))
print(f'Audio files : {n_audio}  (expect 621)')
print(f'VTT files   : {n_vtt}  (expect 600+)')

# overlap sanity: how many audio have a matching vtt (any suffix)
have_vtt = 0
for af in audio_files[:50]:
    vid = os.path.basename(af).replace('.m4a','')
    if glob.glob(f'{VTT_DIR}/{vid}*.vtt'): have_vtt += 1
print(f'VTT match sample : {have_vtt}/50 first audio files have subtitles')

if n_audio == 0:
    raise RuntimeError('No .m4a in chuckle_net_1000/audio — upload audio or fix AUDIO_DIR.')
if have_vtt == 0:
    raise RuntimeError('VTT names do not match audio IDs — check VTT_DIR naming.')
print('\n✅ DIAGNOSTIC PASSED — safe to continue')


In [ ]:
# === VERIFY AUDIO LOADING (v20.1 — robust ffmpeg, never cryptic-crashes) ===
import subprocess, shutil

def ensure_ffmpeg():
    if shutil.which('ffmpeg'): return True
    print('ffmpeg missing — retrying install...')
    if os.system('apt-get install -y ffmpeg > /tmp/apt2.log 2>&1') == 0 and shutil.which('ffmpeg'):
        return True
    print('apt failed — falling back to imageio-ffmpeg static binary')
    r = os.system('pip install -q imageio-ffmpeg > /dev/null 2>&1')
    try:
        import imageio_ffmpeg
        p = imageio_ffmpeg.get_ffmpeg_exe()
        os.makedirs('/usr/local/bin', exist_ok=True)
        shutil.copy(p, '/usr/local/bin/ffmpeg'); os.chmod('/usr/local/bin/ffmpeg', 0o755)
        return bool(shutil.which('ffmpeg'))
    except Exception as e:
        print('fallback failed:', e); return False

if not ensure_ffmpeg():
    raise RuntimeError('ffmpeg unavailable after install + fallback — Runtime → Restart and rerun.')

v = subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True)
print('ffmpeg:', (v.stdout or 'unknown').split(chr(10))[0])

def _test_load(path):
    import wave
    from io import BytesIO
    cmd = ['ffmpeg', '-y', '-i', path, '-ar', '16000', '-ac', '1', '-f', 'wav', '-']
    r = subprocess.run(cmd, capture_output=True)
    if r.returncode != 0 or len(r.stdout) == 0:
        return None, None
    with wave.open(BytesIO(r.stdout), 'rb') as w:
        sr = w.getframerate(); raw = w.readframes(w.getnframes())
    y = np.frombuffer(raw, dtype=np.int16).astype(np.float32) / 32768.0
    return y, sr

if audio_files:
    ok = 0
    for test_audio in audio_files[:3]:
        y, sr = _test_load(test_audio)
        if y is not None and len(y) > 0:
            print(f"  LOAD OK: {os.path.basename(test_audio)} ({len(y)/sr:.1f}s @ {sr}Hz)"); ok += 1
        else:
            print(f"  LOAD FAILED: {os.path.basename(test_audio)}")
    print(f'\nLoader check: {ok}/3 decoded.' + ('' if ok else ' STOP — do not run extraction.'))
else:
    print('No audio files — Cell 2 diagnostic should have caught this; check account/folders.')


In [ ]:
# === VTT PARSING ===

def parse_timestamp(ts):
    ts = ts.strip().replace(',', '.')
    parts = ts.split(':')
    if len(parts) == 3:
        h, m, s = parts
        return float(h)*3600 + float(m)*60 + float(s)
    elif len(parts) == 2:
        m, s = parts
        return float(m)*60 + float(s)
    return float(ts)

def parse_vtt_for_laughter(vtt_path):
    import re
    try:
        with open(vtt_path, 'r', encoding='utf-8') as f:
            content = f.read()
        
        utterances = []
        has_laughter = False
        
        for line in content.split('\n'):
            line = line.strip()
            if '-->' in line:
                if utterances:
                    utterances[-1]['has_laughter'] = has_laughter
                parts = line.split('-->')
                start = parse_timestamp(parts[0])
                end = parse_timestamp(parts[1].strip().split()[0])
                utterances.append({'start': start, 'end': end, 'text': [], 'has_laughter': False})
                has_laughter = False
            elif '[laughter]' in line.lower():
                has_laughter = True
            elif re.search(r'\[(/?(laughter|laugh|laughing|lol|mdr))\]', line, re.IGNORECASE):
                has_laughter = True
            elif utterances and not line.startswith('<'):
                clean = re.sub(r'<[^>]+>', '', line)
                if clean.strip():
                    utterances[-1]['text'].append(clean)
        
        if utterances:
            utterances[-1]['has_laughter'] = has_laughter

        # v20.2: dedup YouTube rolling-caption doubles (each caption appears twice).
        # Merge by (start,end): OR the laughter flags, concat text.
        merged = {}
        for u in utterances:
            k = (round(u['start'], 2), round(u['end'], 2))
            if k in merged:
                merged[k]['has_laughter'] = merged[k]['has_laughter'] or u['has_laughter']
                merged[k]['text'] += u['text']
            else:
                merged[k] = u
        out = sorted(merged.values(), key=lambda u: u['start'])
        return out
    except Exception as e:
        print(f'VTT error: {e}')
        return None

In [ ]:
# === AUDIO LOADING (ffmpeg subprocess — the PROVEN m4a path) ===

import subprocess
import wave
from io import BytesIO

def load_audio(audio_path, sr_target=(16000, 22050)):
    '''Decode ANY format via ffmpeg to 16k mono wav, then resample for 22k.
    NOTE: soundfile/libsndfile CANNOT decode m4a/AAC; librosa>=0.10 has no
    audioread fallback. ffmpeg subprocess is the reliable path in Colab.'''
    try:
        cmd = ['ffmpeg', '-y', '-i', audio_path, '-ar', '16000', '-ac', '1', '-f', 'wav', '-']
        r = subprocess.run(cmd, capture_output=True)
        if r.returncode != 0 or len(r.stdout) == 0:
            return None
        with wave.open(BytesIO(r.stdout), 'rb') as w:
            sr16 = w.getframerate()
            raw = w.readframes(w.getnframes())
        y_16k = np.frombuffer(raw, dtype=np.int16).astype(np.float32) / 32768.0
        if len(y_16k) < sr16:
            return None
        result = {16000: y_16k}
        for sr_t in sr_target:
            if sr_t != 16000:
                result[sr_t] = librosa.resample(y_16k, orig_sr=16000, target_sr=sr_t)
        return result
    except Exception as e:
        print(f'Audio error: {e}')
        return None


In [ ]:
# === PROSODY EXTRACTION ===

def extract_prosody_segment(y, sr, start_s, end_s, hop_length=512):
    start_sample = int(start_s * sr)
    end_sample = int(end_s * sr)
    y_seg = y[start_sample:end_sample]
    if len(y_seg) < sr * 0.1:
        return None
    
    features = []
    rms = librosa.feature.rms(y=y_seg, hop_length=hop_length)[0]
    features.extend([np.mean(rms), np.std(rms), np.max(rms)])
    
    f0, voiced, prob = librosa.pyin(y_seg, fmin=50, fmax=500, sr=sr, hop_length=hop_length)
    f0 = np.nan_to_num(f0, nan=0)
    f0_valid = f0[f0 > 0]
    if len(f0_valid) > 0:
        features.extend([np.mean(f0_valid), np.std(f0_valid), np.max(f0_valid)-np.min(f0_valid)])
    else:
        features.extend([0, 0, 0])
    
    zcr = librosa.feature.zero_crossing_rate(y_seg, hop_length=hop_length)[0]
    features.extend([np.mean(zcr), np.std(zcr)])
    
    sc = librosa.feature.spectral_centroid(y=y_seg, sr=sr, hop_length=hop_length)[0]
    features.extend([np.mean(sc), np.std(sc)])
    
    sb = librosa.feature.spectral_bandwidth(y=y_seg, sr=sr, hop_length=hop_length)[0]
    features.extend([np.mean(sb), np.std(sb)])
    
    rolloff = librosa.feature.spectral_rolloff(y=y_seg, sr=sr, hop_length=hop_length)[0]
    features.extend([np.mean(rolloff), np.std(rolloff)])
    
    mfcc = librosa.feature.mfcc(y=y_seg, sr=sr, n_mfcc=13, hop_length=hop_length)
    for i in range(13):
        features.append(np.mean(mfcc[i]))
    
    return np.array(features, dtype=np.float32)

In [ ]:
# === WAVLM SETUP ===

from transformers import Wav2Vec2Model

print('Loading WavLM...')
wavlm = Wav2Vec2Model.from_pretrained('microsoft/wavlm-base')
wavlm.to(DEVICE)
wavlm.eval()
print(f'WavLM loaded on {DEVICE}')

def extract_wavlm_segment(y_16k, start_s, end_s):
    start_sample = int(start_s * 16000)
    end_sample = int(end_s * 16000)
    y_seg = y_16k[start_sample:end_sample]
    if len(y_seg) < 1600:
        return None
    with torch.no_grad():
        inputs = torch.FloatTensor(y_seg).unsqueeze(0).to(DEVICE)
        am = torch.ones_like(inputs)
        out = wavlm(inputs, attention_mask=am)
        emb = out.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
    return emb.astype(np.float32)

In [ ]:
# === MAIN EXTRACTION ===

GATE_IDS = ['37ae9Uj_Qk0','1Nb3_os4RSA','4IWPavFgVn0','2xAXTiPInbo','40uGUxg1_Xg',
            '3eOkj9uCfT0','24Z7jOfmFmM','2vfE1x9US7w','0M9yRpzR85U','45VWTm3ldJ8',
            '26nQZ9pgF9U','2XdRwA-hPsA','15biDQyQN4A','0jOJis6Oj1s','0pXpuCyywhA',
            '0-cy8J2foWU','1DEm4GuUtSQ','3tpGSUi2KaA','3pYQfnggvYY','BfNHCavFtzQ']  # marker-rich 20 (full-620 forensics Sep 8)
if GATE_N and GATE_N > 0:
    have = {os.path.basename(af).replace('.m4a','') for af in audio_files}
    missing = set(GATE_IDS) - have
    if missing: print(f'WARNING: gate IDs missing from audio dir: {missing}')
    audio_files = [af for af in audio_files if os.path.basename(af).replace('.m4a','') in GATE_IDS]
    print(f'GATE MODE: {len(audio_files)} curated marker-rich videos')

CHECKPOINT_FILE = f'{OUTPUT}/extraction_checkpoint_v21.npz'
CHECKPOINT_IDX = f'{OUTPUT}/processed_idx_v21.txt'

processed_idx = set()
if os.path.exists(CHECKPOINT_IDX):
    with open(CHECKPOINT_IDX, 'r') as f:
        processed_idx = set(f.read().splitlines())
    print(f'Resuming from {len(processed_idx)} files')

def save_checkpoint(features, labels, vids, starts, ends):
    np.savez_compressed(CHECKPOINT_FILE, features=features, labels=labels, vids=np.array(vids),
                        utt_starts=np.array(starts), utt_ends=np.array(ends))
    with open(CHECKPOINT_IDX, 'w') as f:
        f.write('\n'.join(sorted(processed_idx)))
    print(f'Checkpoint: {len(features)} samples at {len(processed_idx)} files')

def load_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        data = np.load(CHECKPOINT_FILE, allow_pickle=True)
        if len(data['features']) > 0:
            st = list(data['utt_starts']) if 'utt_starts' in data else [None]*len(data['features'])
            en = list(data['utt_ends']) if 'utt_ends' in data else [None]*len(data['features'])
            return data['features'], data['labels'], list(data['vids']), st, en
    return None, None, None, None, None

start_idx = 0
if processed_idx:
    for i, af in enumerate(audio_files):
        vid = os.path.basename(af).replace('.m4a', '')
        if vid not in processed_idx:
            start_idx = i
            break
    else:
        print(f'All {len(audio_files)} files done!')

print(f'Starting from index {start_idx}/{len(audio_files)}')

# Pre-flight check: abort loudly if no audio files found
if not audio_files:
    raise RuntimeError(f'ERROR: audio_files is empty! Check that Cell 1 ran and data exists at {AUDIO_DIR}.')
print(f'Processing {len(audio_files)} audio files...')


all_features, all_labels, all_vids = [], [], []
all_starts, all_ends = [], []
stats = {'no_vtt': 0, 'no_audio': 0, 'no_utt': 0, 'bad_feat': 0}
existing = load_checkpoint()
# Load checkpoint whenever resume mode (processed_idx non-empty).
# A truly fresh run has empty processed_idx -> no stale data loaded.
if existing[0] is not None and len(existing[0]) > 0 and len(processed_idx) > 0:
    all_features, all_labels, all_vids = list(existing[0]), list(existing[1]), list(existing[2])
    all_starts, all_ends = list(existing[3]), list(existing[4])
    print(f'Resuming with {len(all_features)} samples from checkpoint')

for i, af in enumerate(tqdm(audio_files[start_idx:], desc='Processing')):
    vid = os.path.basename(af).replace('.m4a', '')
    if vid in processed_idx:
        continue
    
    vtt_path = None
    for p in [f'{VTT_DIR}/{vid}.vtt', f'{VTT_DIR}/{vid}.en.vtt', f'{VTT_DIR}/{vid}.en-US.vtt']:
        if os.path.exists(p):
            vtt_path = p
            break
    if not vtt_path:
        matches = glob.glob(f'{VTT_DIR}/{vid}*.vtt')
        if matches:
            vtt_path = matches[0]
    if not vtt_path:
        stats['no_vtt'] += 1
        continue
    
    utterances = parse_vtt_for_laughter(vtt_path)
    if not utterances:
        stats['no_utt'] += 1
        continue
    
    audio_data = load_audio(af)
    if audio_data is None:
        stats['no_audio'] += 1
        continue
    
    y_16k = audio_data[16000]
    y_22k = audio_data[22050]
    
    for utt in utterances:
        prosody = extract_prosody_segment(y_22k, 22050, utt['start'], utt['end'])
        wavlm_emb = extract_wavlm_segment(y_16k, utt['start'], utt['end'])
        if prosody is not None and wavlm_emb is not None:
            all_features.append(np.concatenate([wavlm_emb, prosody]))
            all_labels.append(1 if utt['has_laughter'] else 0)
            all_vids.append(vid)
            all_starts.append(float(utt['start'])); all_ends.append(float(utt['end']))
        else:
            stats['bad_feat'] += 1
    
    print(f"  {vid}: {len(utterances)} utts, {sum(1 for u in utterances if u['has_laughter'])} marker-pos")
    processed_idx.add(vid)
    if len(processed_idx) % max(1, FILES_PER_SAVE) == 0:
        save_checkpoint(np.array(all_features), np.array(all_labels), all_vids, all_starts, all_ends)

if all_features:
    save_checkpoint(np.array(all_features), np.array(all_labels), all_vids, all_starts, all_ends)

print(f'\nExtracted: {len(all_features)} samples')
y_arr = np.array(all_labels)
if len(y_arr) > 0:
    print(f'Positive: {y_arr.sum()} ({100*y_arr.mean():.1f}%)')
print(f'Skipped — no_vtt:{stats["no_vtt"]}, no_audio:{stats["no_audio"]}, no_utt:{stats["no_utt"]}, bad_feat:{stats["bad_feat"]}')

# === v20 HARD RULE: pos_rate gate ===
y_arr = np.array(all_labels)
pos_rate = float(y_arr.mean()) if len(y_arr) else 0.0
if pos_rate < 0.005:  # FAIL line lowered Sep 8: forensics PROVED full-set 1.16% pos is genuine (docs/GATE_FORENSICS_620V_README.md); old 2% line would false-fail the real 620v run. <0.5% = only truly broken labels
    raise RuntimeError(f'HARD FAIL: {pos_rate*100:.2f}% < 2% — labels BROKEN. Check parsing/mix. DO NOT TRAIN.')
elif pos_rate < 0.10:
    print(f'WARN: pos_rate {pos_rate*100:.2f}% in 2-10% band. Labels VERIFIED genuine — the 620v collection is '
          f'marker-poor (forensics Sep 8: full-set 1.16%, only 174/620 videos carry markers). Proceeding with '
          f'pos_weight + PR-AUC; RICH subset trained separately below.')
else:
    print(f'pos_rate gate PASSED: {pos_rate*100:.1f}%')


In [ ]:
# === SAVE FEATURES ===

if not all_features:
    raise RuntimeError('ERROR: No features extracted! Check skip stats above.')

np.savez_compressed(f'{OUTPUT}/utterance_features_v21.npz',
                    features=np.array(all_features),
                    labels=np.array(all_labels),
                    vids=np.array(all_vids),
                    utt_starts=np.array(all_starts),
                    utt_ends=np.array(all_ends))
print(f'Saved: {OUTPUT}/utterance_features_v21.npz ({len(all_features)} samples)')


In [ ]:
# === DUAL TRAINING (v20.2): FULL set + RICH subset — F1, PR-AUC, IoU-F1@0.2 ===
import torch, torch.nn as nn, numpy as np, json, collections
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, precision_score, recall_score, average_precision_score
from sklearn.preprocessing import StandardScaler
import random

torch.manual_seed(42); random.seed(42); np.random.seed(42)

data = np.load(f'{OUTPUT}/utterance_features_v21.npz')
X, y, vids = data['features'], data['labels'].astype(int), data['vids']
starts, ends = data['utt_starts'], data['utt_ends']
print(f'Data: {len(y)} samples, {len(set(vids))} videos, pos={int(y.sum())} ({100*y.mean():.2f}%)')

class FusionMLP(nn.Module):
    def __init__(self, dim, hidden=[512,256,64]):
        super().__init__()
        self.bn0 = nn.BatchNorm1d(dim); layers=[]; prev=dim
        for h in hidden:
            layers += [nn.Linear(prev,h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(0.3)]; prev=h
        layers.append(nn.Linear(prev,1)); self.net=nn.Sequential(*layers)
    def forward(self,x): return self.net(self.bn0(x)).squeeze(-1)

def merge_iv(ivs, gap=0.8):
    if not ivs: return []
    ivs = sorted(ivs); out=[list(ivs[0])]
    for s,e in ivs[1:]:
        if s - out[-1][1] <= gap: out[-1][1] = max(out[-1][1], e)
        else: out.append([s,e])
    return [(s,e) for s,e in out]

def iou_interval(a, b):
    inter = min(a[1],b[1]) - max(a[0],b[0])
    if inter <= 0: return 0.0
    union = (a[1]-a[0]) + (b[1]-b[0]) - inter
    return inter/union

def iou_f1_interval(vlist, yt, yp, st, en, thr=0.2):
    tps=fps=fns=0
    for v in set(vlist):
        idx=[i for i,vv in enumerate(vlist) if vv==v]
        t_iv=merge_iv([(float(st[i]),float(en[i])) for i in idx if yt[i]==1])
        p_iv=merge_iv([(float(st[i]),float(en[i])) for i in idx if yp[i]==1])
        mt,mp=set(),set()
        pairs=sorted(((iou_interval(a,b),j,k) for j,a in enumerate(t_iv) for k,b in enumerate(p_iv)),
                     key=lambda x:-x[0])
        for val,j,k in pairs:
            if val < thr: break
            if j not in mt and k not in mp:
                mt.add(j); mp.add(k); tps+=1
        fps += len(p_iv)-len(mp); fns += len(t_iv)-len(mt)
    prec=tps/max(1,tps+fps); rec=tps/max(1,tps+fns)
    return 2*prec*rec/max(1e-9,prec+rec), prec, rec

def train_eval(X, y, vids, starts, ends, tag):
    scaler = StandardScaler(); X_s = scaler.fit_transform(X)
    gkf = GroupKFold(n_splits=5)
    seg_f1s, seg_p, seg_r, praucs, ious = [], [], [], [], []
    last_model, last_scaler = None, scaler
    for fold,(tr,te) in enumerate(gkf.split(X_s, y, vids)):
        model = FusionMLP(X_s.shape[1]).to(DEVICE)
        opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
        Xtr = torch.FloatTensor(X_s[tr]); ytr = torch.FloatTensor(y[tr])
        pw = torch.tensor([max(1.0,(1-ytr.mean())/max(0.01,ytr.mean()))]).to(DEVICE)
        lf = nn.BCEWithLogitsLoss(pos_weight=pw)
        n_tr = len(Xtr)-1 if len(Xtr)%32==1 else len(Xtr)
        for ep in range(100):
            model.train()
            for i in range(0, n_tr, 32):
                bx, by = Xtr[i:i+32].to(DEVICE), ytr[i:i+32].to(DEVICE)
                opt.zero_grad(); loss = lf(model(bx), by); loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
        model.eval()
        with torch.no_grad():
            pr = torch.sigmoid(model(torch.FloatTensor(X_s[te]).to(DEVICE))).cpu().numpy()
        pb = (pr>0.5).astype(int); yt = y[te]
        seg_f1s.append(f1_score(yt, pb, zero_division=0))
        seg_p.append(precision_score(yt, pb, zero_division=0))
        seg_r.append(recall_score(yt, pb, zero_division=0))
        praucs.append(float(average_precision_score(yt, pr)) if yt.sum()>0 else 0.0)
        iv = iou_f1_interval(list(np.array(vids)[te]), yt, pb, np.array(starts)[te], np.array(ends)[te])
        ious.append(iv[0])
        print(f'  [{tag}] fold {fold+1}: F1={seg_f1s[-1]:.4f} PR-AUC={praucs[-1]:.4f} IoU-F1@0.2={iv[0]:.4f}')
        last_model = model
    r = {'tag': tag, 'n_samples': int(len(y)), 'n_videos': int(len(set(vids))), 'pos_rate': float(y.mean()),
         'seg_f1': {'mean': float(np.mean(seg_f1s)), 'std': float(np.std(seg_f1s))},
         'precision': float(np.mean(seg_p)), 'recall': float(np.mean(seg_r)),
         'pr_auc': {'mean': float(np.mean(praucs)), 'std': float(np.std(praucs))},
         'iou_f1_02': {'mean': float(np.mean(ious)), 'std': float(np.std(ious)), 'merge_gap_s': 0.8}}
    print(f"[{tag}] MEAN: F1={r['seg_f1']['mean']:.4f}  PR-AUC={r['pr_auc']['mean']:.4f}  IoU-F1@0.2={r['iou_f1_02']['mean']:.4f}")
    return r, last_model, last_scaler

# per-video pos stats (on extracted samples) -> RICH subset
vpos, vtot = collections.Counter(), collections.Counter()
for v, lab in zip(vids, y):
    vtot[v] += 1; vpos[v] += int(lab)
rich_vids = {v for v in vtot if vtot[v] > 0 and 100*vpos[v]/vtot[v] >= RICH_MIN_PCT and vpos[v] >= RICH_MIN_POS}
print(f'RICH subset (pos_pct>={RICH_MIN_PCT}%, pos>={RICH_MIN_POS}): {len(rich_vids)} videos')
rich_mask = np.array([v in rich_vids for v in vids])

print('\n=== RUN 1/2: FULL set ===')
res_full, model_full, scaler_full = train_eval(X, y, vids, starts, ends, 'FULL')
print('\n=== RUN 2/2: RICH subset ===')
if rich_mask.sum() >= 100:
    res_rich, model_rich, scaler_rich = train_eval(X[rich_mask], y[rich_mask], vids[rich_mask],
                                                   starts[rich_mask], ends[rich_mask], 'RICH')
    torch.save({'model_state_dict': model_rich.state_dict(), 'scaler': scaler_rich, 'res': res_rich},
               f'{OUTPUT}/fusion_model_rich_v21.pt')
else:
    res_rich = {'tag': 'RICH', 'skipped': 'subset too small in gate mode'}
torch.save({'model_state_dict': model_full.state_dict(), 'scaler': scaler_full, 'res': res_full},
           f'{OUTPUT}/fusion_model_full_v21.pt')
print('\nModels saved to', OUTPUT)


In [ ]:
# === v20.2: RESULTS JSON (provenance: dual run + label forensics) ===
import datetime, sys, transformers
results = {
 'schema': 'chucklenet_v20.2', 'timestamp_utc': datetime.datetime.utcnow().isoformat(),
 'label_tier': 'TIER-1 weak: VTT caption laughter markers — report as caption-marker detection',
 'label_forensics_sep8': {
    'full_set_pos_rate': 0.0116, 'full_set_utts': 243501, 'full_set_pos': 2816,
    'videos_with_markers': '174/620',
    'vtt_source_verified': 'Drive VTTs byte-identical to curated local label set',
    'parser_verified': True,
    'youtube_dedup': 'rolling-caption doubles merged (OR flags) — parsed counts were ~50% inflated before'},
 'pos_gate': {'fail_below': 0.005, 'warn_band': '2-10%',
              'note': '10% = clean pass; WARN = genuine-but-sparse (proven Sep 8 forensics); <0.5% = broken (line lowered Sep 8: full-set genuine rate is 1.16%)'},
 'anchor_rule': 'Gold claims require Gillick-162v / StandUp4AI anchor evals (docs/LABEL_HIERARCHY.md)',
 'config': {'gate_n': GATE_N, 'features': 'WavLM-768+prosody-23', 'model': 'FusionMLP 512-256-64 seed42',
            'split': 'GroupKFold5 video-level',
            'rich_filter': {'min_pos_pct': RICH_MIN_PCT, 'min_pos': RICH_MIN_POS}},
 'runs': {'FULL': res_full, 'RICH': res_rich},
 'env': {'torch': torch.__version__, 'transformers': transformers.__version__, 'python': sys.version.split()[0]},
 'references': {'naive_IoU': 0.29, 'standup4ai_118v_best': 0.3302, 'standup4ai_baseline': 0.51,
                'gillick_gold': 0.559}
}
with open(f'{OUTPUT}/results_v21.json','w') as f:
    json.dump(results, f, indent=2)
print(json.dumps(results, indent=2))
print('\nSaved:', f'{OUTPUT}/results_v21.json', '— paste back for the decision graph')
